# 04 — Train: MLP regression

Searches an MLP (multilayer perceptron) architecture and regularization strength for the configured target station's direct 24-hour water-level forecast over the joined feature artifacts, then evaluates the selected model once on the sealed test cohort.

**Inputs:** joined train/test feature artifacts and their metadata contract
**Outputs:** in-notebook prediction preview/test metrics, an MLflow run hierarchy, and the selected model plus manifest in `models/`

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

Imports the dependencies and loads the joined feature metadata. Its predictor columns are the source of truth for the model inputs: target-station engineered features plus raw measurements from every retained station at issue time `t`. The MLP architecture/alpha search and validation policy are explicit constants so every fold and MLflow run remains inspectable.

Unlike Ridge, `MLPRegressor` is a stochastic estimator (weight initialization, `adam`'s optimization path), so `RANDOM_STATE` is fixed and reused in every fold fit and the final retrain, so the manifest remains a reproducible record. There is also no log1p axis: both predictors and the target are `StandardScaler`-scaled (target scaling via a `TransformedTargetRegressor` wrapping `StandardScaler`), because MLP's gradient-based training needs a zero-centered, unit-scale target to converge, unlike Ridge's closed-form solve which doesn't care about target scale.

**Parameters**

| Parameter | Value | What it does |
| --- | --- | --- |
| `PROCESSED_DIR` | `data/processed/joined` | Directory the joined Stage-3 Parquets and metadata are read from. |
| `PREDICTION_PREVIEW_ROWS` | `5` | Number of scored test rows shown in the final preview. |
| `FULL_FEATURE_COLUMNS` | all metadata-declared predictors | The complete predictor contract used for common eligibility; raw timestamps and metadata fields are not model inputs. |
| `FEATURE_SUBSETS` | six predefined subsets | Candidate feature lists derived from the full metadata contract in metadata order. |
| `TARGET_COLUMNS` | `{TARGET_STATION_ID}__target_t_plus_01` … `{TARGET_STATION_ID}__target_t_plus_24` | The 24 future water levels predicted directly from one issue-time feature vector. |
| `FORECAST_HORIZON_HOURS` | `24` | Number of direct future target outputs and the metadata contract width. |
| `MLP_HIDDEN_LAYER_SIZES` | `[(32,), (64,), (128,), (64, 32)]` | Candidate hidden-layer architectures. |
| `MLP_ALPHAS` | `[0.0001, 0.001, 0.01, 0.1]` | Candidate L2 regularization strengths. |
| `MLP_ACTIVATION` | `"relu"` | Fixed hidden-layer activation function. |
| `MLP_SOLVER` | `"adam"` | Fixed weight-optimization solver. |
| `MLP_LEARNING_RATE_INIT` | `0.001` | Fixed initial learning rate. |
| `MLP_MAX_ITER` | `300` | Fixed maximum training iterations. |
| `MLP_BATCH_SIZE` | `"auto"` | Fixed minibatch size. |
| `MLP_EARLY_STOPPING` | `False` | Whether training may stop before `MLP_MAX_ITER` on a validation split; fixed off so every candidate trains for a comparable, inspectable iteration budget. |
| `RANDOM_STATE` | `src.config.RANDOM_STATE` | Fixed seed for weight initialization and the solver's stochastic optimization path, reused in every fold fit and the final retrain. |
| `N_VALIDATION_FOLDS` | `5` | Number of expanding-window validation folds. |
| `INITIAL_TRAIN_FRACTION` | `0.50` | Approximate fraction of eligible rows in the first fold's training window. |
| `EMBARGO_HOURS` | `24` | Number of rows left between each fold's training and validation windows. |
| `CV_SELECTION_METRIC` | `"rmse"` | Aggregate CV metric used to select the candidate; `"mae"` is also supported. |
| `MLFLOW_EXPERIMENT_NAME` | `"mlp"` | Experiment receiving the parent, nested fold, and final test runs. |
| `MODEL_PATH` | `models/mlp_{TARGET_STATION_ID}.joblib` | Bundled preprocessing and selected `TransformedTargetRegressor` trained on all eligible training rows. |
| `MODEL_METADATA_PATH` | `models/mlp_{TARGET_STATION_ID}.json` | Schema-1.0 manifest recording the feature contract, selected subset/hidden_layer_sizes/alpha, the full CV candidate table, the aggregate and per-horizon sealed-test metrics, and the training range. Written once, after the sealed test. |

## Joint feature-subset, hidden_layer_sizes, and alpha search

The notebook compares one global `(feature subset, hidden_layer_sizes, alpha)` triple with five expanding-window folds. The six predefined subsets are the same ones used by `04_02_train_ridge.ipynb`, derived from metadata-declared predictors and retaining their metadata order:

| Subset | Intended predictors | Current size |
| --- | --- | ---: |
| `full` | All declared predictors | 81 |
| `all_station_hydrology_quality_time` | Water-level history, imputation indicators, and calendar signals for every station; excludes weather | 55 |
| `raw_all_stations` | Current `water_level`, `imputed`, precipitation, and temperature for every station | 32 |
| `target_station_full` | All declared predictors for the target station only | 53 |
| `target_station_hydrology_quality_time` | Target-station water-level history, imputation indicators, and calendar signals | 41 |
| `current_water_levels_all_stations` | Current `water_level` for every station | 8 |

The full contract determines eligibility once for both artifacts, so all candidates use the same eligible rows and identical fold indices. The search performs `6 × 4 × 4 × 5 = 480` fold fits, then retrains only the selected candidate and evaluates the sealed test once — noticeably more fits than Ridge's `6 × 5 × 2 × 5 = 300`, so this notebook takes materially longer to run.

In [ ]:
import json
from pathlib import Path
from uuid import uuid4

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from IPython.display import display
from joblib import dump

from src.config import (
    CV_SELECTION_METRIC,
    EMBARGO_HOURS,
    FORECAST_HORIZON_HOURS,
    INITIAL_TRAIN_FRACTION,
    MLFLOW_TRACKING_URI,
    N_VALIDATION_FOLDS,
    RANDOM_STATE,
    TARGET_STATION_ID,
    WEATHER_VARIABLES,
)
from src.dataset import load_joined_dataset
from src.metrics import metric_tables
from src.plots import (
    cv_error_boxplots_figure,
    predicted_vs_actual_figure,
    test_error_boxplots_figure,
)
from src.training import (
    numeric_predictors,
    prediction_preview,
    summarize_cv_metrics,
    validate_predictions,
)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
NOTEBOOK_EXECUTION_UUID = str(uuid4())
PROCESSED_DIR = Path("data/processed/joined")
METADATA_PATH = PROCESSED_DIR / "all_stations_feature_metadata.json"
train_path = PROCESSED_DIR / "all_stations_train_features.parquet"
test_path = PROCESSED_DIR / "all_stations_test_features.parquet"
MLP_HIDDEN_LAYER_SIZES = [(32,), (64,), (128,), (64, 32)]
MLP_ALPHAS = [0.0001, 0.001, 0.01, 0.1]
MLP_ACTIVATION = "relu"
MLP_SOLVER = "adam"
MLP_LEARNING_RATE_INIT = 0.001
MLP_MAX_ITER = 500
MLP_BATCH_SIZE = "auto"
MLP_EARLY_STOPPING = False
MLFLOW_EXPERIMENT_NAME = "mlp"
PREDICTION_PREVIEW_ROWS = 5
if CV_SELECTION_METRIC not in {"mae", "rmse"}:
    raise ValueError("CV_SELECTION_METRIC must be either 'mae' or 'rmse'")
station_id = TARGET_STATION_ID
MODEL_DIR = Path("models")
MODEL_PATH = MODEL_DIR / f"mlp_{station_id}.joblib"
MODEL_METADATA_PATH = MODEL_DIR / f"mlp_{station_id}.json"

## Shared evaluation cohort

The model is fit and scored on rows from the joined feature artifacts. One row is one timestamp `t`, and it qualifies only when both conditions hold:

1. **Stage 3 marked the future window valid.** `{TARGET_STATION_ID}__target_valid` is true, and all 24 `{TARGET_STATION_ID}__target_t_plus_01` … `{TARGET_STATION_ID}__target_t_plus_24` values are present.
2. **Every model input is present.** All full-contract predictors must be available: the target station's engineered features plus every retained station's raw water level, imputation flag, precipitation, and temperature at issue time `t`.

Train and test are filtered independently and are never pooled: the test artifact is sealed, and no statistic used by the model — not even a scaler mean — is ever computed from it. Eligibility is deliberately based on `FULL_FEATURE_COLUMNS`, not a candidate subset, so all candidates compare the same cohort.

## Shared helpers

The joined dataset — contract loading, common-cohort preparation, ordered feature subsets, and chronological folds — comes from `src.dataset`. Prediction checks, metric summaries, and previews come from `src.training`, and the evaluation figures from `src.plots`. MLP candidate ranking remains local because its subset/architecture/alpha tie-breaking policy is estimator-specific.

In [ ]:
from src.mlp import (
    build_mlp_estimator,
    format_hidden_layer_sizes,
    save_mlp_manifest,
    select_candidate,
)

## Load the joined dataset

`load_joined_dataset()` does the whole preamble in one call: it reads the joined feature metadata, `all_stations_train_features.parquet`, and `all_stations_test_features.parquet` from the Stage-3 directory and checks their station, horizon, and column contracts, so a missing or incompatible artifact fails before any model work begins.

It then applies the eligibility cohort to each artifact independently — keeping only target-valid rows with complete predictors and targets, sorted chronologically — derives the six ordered feature subsets, and builds the expanding validation folds. If either split has no eligible row, or the folds violate the configured CV policy, the notebook stops here rather than fitting on an empty frame or reporting a metric computed from nothing.

In [ ]:
dataset = load_joined_dataset(
    METADATA_PATH,
    train_path,
    test_path,
    station_id=station_id,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
    weather_variables=WEATHER_VARIABLES,
    initial_train_fraction=INITIAL_TRAIN_FRACTION,
    n_validation_folds=N_VALIDATION_FOLDS,
    embargo_rows=EMBARGO_HOURS,
)
contract = dataset.contract
TARGET_COLUMNS = list(contract.target_columns)
FULL_FEATURE_COLUMNS = list(contract.predictor_columns)
FEATURE_SUBSETS = dataset.feature_subsets
train_rows = dataset.train_rows
test_rows = dataset.test_rows
INPUT_PARQUET_SHA256_PARAMS = dataset.input_hashes

## Joint time-series subset, hidden_layer_sizes, and alpha search

Eligible training rows are sorted by issue time before `TimeSeriesSplit` creates five expanding-window folds. The explicit `test_size` allocates the post-initial-training portion across the folds, while the 24-row gap acts as the requested hourly embargo. Each `(subset, hidden_layer_sizes, alpha)` candidate has one MLflow parent and each fold has one nested child run: `6 × 4 × 4 × 5 = 480` fits. The current execution must produce the complete 96-candidate Cartesian product before selection.

Every fold builds its own preprocessing-plus-MLP estimator via `build_mlp_estimator` — a `StandardScaler`-scaled `MLPRegressor` wrapped in a `TransformedTargetRegressor` that also scales the target — using only that fold's training rows and the candidate's explicit columns, and the fixed `RANDOM_STATE` so every fold's weight initialization and optimization path is reproducible. The sealed test cohort is not referenced until the final fit below.

In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
cv_splits = dataset.folds
validation_test_size = dataset.validation_test_size
cv_results_rows = []
cv_horizon_rows_by_candidate = {}
expected_candidate_keys = {
    (subset_name, format_hidden_layer_sizes(hidden_layer_sizes), float(alpha))
    for subset_name in FEATURE_SUBSETS
    for hidden_layer_sizes in MLP_HIDDEN_LAYER_SIZES
    for alpha in MLP_ALPHAS
}

for subset_name, feature_columns in FEATURE_SUBSETS.items():
    for hidden_layer_sizes in MLP_HIDDEN_LAYER_SIZES:
        hidden_layer_sizes_label = format_hidden_layer_sizes(hidden_layer_sizes)
        for alpha in MLP_ALPHAS:
            fold_aggregate_rows = []
            fold_horizon_rows = []
            with mlflow.start_run(
                run_name=f"mlp_cv_{subset_name}_{hidden_layer_sizes_label}_alpha_{alpha:g}",
                nested=False,
                tags={
                    "phase": "cv",
                    "run_type": "candidate_parent",
                    "subset": subset_name,
                    "hidden_layer_sizes": hidden_layer_sizes_label,
                    "execution_uuid": NOTEBOOK_EXECUTION_UUID,
                },
            ):
                mlflow.log_params(
                    {
                        "phase": "cv",
                        "run_type": "candidate_parent",
                        "subset": subset_name,
                        "feature_count": len(feature_columns),
                        "feature_columns": json.dumps(feature_columns),
                        **INPUT_PARQUET_SHA256_PARAMS,
                        "alpha": alpha,
                        "hidden_layer_sizes": hidden_layer_sizes_label,
                        "activation": MLP_ACTIVATION,
                        "solver": MLP_SOLVER,
                        "learning_rate_init": MLP_LEARNING_RATE_INIT,
                        "max_iter": MLP_MAX_ITER,
                        "batch_size": str(MLP_BATCH_SIZE),
                        "early_stopping": MLP_EARLY_STOPPING,
                        "random_state": RANDOM_STATE,
                        "n_validation_folds": N_VALIDATION_FOLDS,
                        "validation_test_size": validation_test_size,
                        "embargo_hours": EMBARGO_HOURS,
                        "selection_metric": CV_SELECTION_METRIC,
                        "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
                        "common_train_rows": len(train_rows),
                    }
                )

                for fold_number, (
                    fold_train_indices,
                    fold_validation_indices,
                ) in enumerate(cv_splits, start=1):
                    fold_train_rows = train_rows.iloc[fold_train_indices]
                    fold_validation_rows = train_rows.iloc[fold_validation_indices]
                    fold_model = build_mlp_estimator(
                        feature_columns,
                        hidden_layer_sizes=hidden_layer_sizes,
                        alpha=alpha,
                        activation=MLP_ACTIVATION,
                        solver=MLP_SOLVER,
                        learning_rate_init=MLP_LEARNING_RATE_INIT,
                        max_iter=MLP_MAX_ITER,
                        batch_size=MLP_BATCH_SIZE,
                        early_stopping=MLP_EARLY_STOPPING,
                        random_state=RANDOM_STATE,
                    )
                    fold_model.fit(
                        numeric_predictors(fold_train_rows, feature_columns),
                        fold_train_rows[TARGET_COLUMNS],
                    )
                    fold_predictions = validate_predictions(
                        fold_model.predict(
                            numeric_predictors(fold_validation_rows, feature_columns)
                        ),
                        expected_rows=len(fold_validation_rows),
                        target_columns=TARGET_COLUMNS,
                        artifact_name="fold",
                    )

                    fold_aggregate, fold_per_horizon = metric_tables(
                        fold_validation_rows[TARGET_COLUMNS],
                        fold_predictions,
                        target_columns=TARGET_COLUMNS,
                        station_id=station_id,
                    )
                    fold_aggregate_rows.append(fold_aggregate.iloc[0])
                    fold_horizon_rows.append(fold_per_horizon)
                    with mlflow.start_run(
                        run_name=f"mlp_cv_{subset_name}_{hidden_layer_sizes_label}_alpha_{alpha:g}_fold_{fold_number}",
                        nested=True,
                        tags={
                            "phase": "cv",
                            "run_type": "fold",
                            "subset": subset_name,
                            "hidden_layer_sizes": hidden_layer_sizes_label,
                            "fold": str(fold_number),
                            "execution_uuid": NOTEBOOK_EXECUTION_UUID,
                        },
                    ):
                        mlflow.log_params(
                            {
                                "phase": "cv",
                                "run_type": "fold",
                                "subset": subset_name,
                                "feature_count": len(feature_columns),
                                **INPUT_PARQUET_SHA256_PARAMS,
                                "alpha": alpha,
                                "hidden_layer_sizes": hidden_layer_sizes_label,
                                "fold": fold_number,
                                "train_rows": len(fold_train_rows),
                                "validation_rows": len(fold_validation_rows),
                                "gap_rows": EMBARGO_HOURS,
                                "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
                                "train_start": fold_train_rows["timestamp"]
                                .iloc[0]
                                .isoformat(),
                                "train_end": fold_train_rows["timestamp"]
                                .iloc[-1]
                                .isoformat(),
                                "validation_start": fold_validation_rows["timestamp"]
                                .iloc[0]
                                .isoformat(),
                                "validation_end": fold_validation_rows["timestamp"]
                                .iloc[-1]
                                .isoformat(),
                                "train_index_start": int(fold_train_indices[0]),
                                "train_index_end": int(fold_train_indices[-1]),
                                "validation_index_start": int(
                                    fold_validation_indices[0]
                                ),
                                "validation_index_end": int(
                                    fold_validation_indices[-1]
                                ),
                            }
                        )
                        mlflow.log_metrics(
                            {
                                "fold_mae": float(fold_aggregate.iloc[0]["mae"]),
                                "fold_rmse": float(fold_aggregate.iloc[0]["rmse"]),
                                "fold_me": float(fold_aggregate.iloc[0]["me"]),
                                "fold_r2": float(fold_aggregate.iloc[0]["r2"]),
                                **{
                                    f"fold_mae_horizon_{row.horizon_hours:02d}": float(
                                        row.mae
                                    )
                                    for row in fold_per_horizon.itertuples()
                                },
                                **{
                                    f"fold_me_horizon_{row.horizon_hours:02d}": float(
                                        row.me
                                    )
                                    for row in fold_per_horizon.itertuples()
                                },
                                **{
                                    f"fold_r2_horizon_{row.horizon_hours:02d}": float(
                                        row.r2
                                    )
                                    for row in fold_per_horizon.itertuples()
                                },
                                **{
                                    f"fold_rmse_horizon_{row.horizon_hours:02d}": float(
                                        row.rmse
                                    )
                                    for row in fold_per_horizon.itertuples()
                                },
                            }
                        )

                fold_aggregate_metrics = pd.DataFrame(fold_aggregate_rows)
                fold_horizon_metrics = pd.concat(fold_horizon_rows, ignore_index=True)
                parent_metrics = summarize_cv_metrics(
                    fold_aggregate_metrics,
                    fold_horizon_metrics,
                )
                candidate_key = (subset_name, hidden_layer_sizes_label, float(alpha))
                cv_horizon_rows_by_candidate[candidate_key] = fold_horizon_rows.copy()
                mlflow.log_metrics(parent_metrics)
                cv_results_rows.append(
                    {
                        "subset": subset_name,
                        "feature_count": len(feature_columns),
                        "hidden_layer_sizes": hidden_layer_sizes_label,
                        "alpha": float(alpha),
                        "mae_mean": parent_metrics["cv_mae_mean"],
                        "mae_std": parent_metrics["cv_mae_std"],
                        "rmse_mean": parent_metrics["cv_rmse_mean"],
                        "rmse_std": parent_metrics["cv_rmse_std"],
                        "me_mean": parent_metrics["cv_me_mean"],
                        "me_std": parent_metrics["cv_me_std"],
                        "r2_mean": parent_metrics["cv_r2_mean"],
                        "r2_std": parent_metrics["cv_r2_std"],
                        **{
                            metric_name: metric_value
                            for metric_name, metric_value in parent_metrics.items()
                            if metric_name
                            not in {
                                "cv_mae_mean",
                                "cv_mae_std",
                                "cv_rmse_mean",
                                "cv_rmse_std",
                                "cv_me_mean",
                                "cv_me_std",
                                "cv_r2_mean",
                                "cv_r2_std",
                            }
                        },
                    }
                )

cv_experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
if cv_experiment is None:
    raise ValueError(f"MLflow experiment {MLFLOW_EXPERIMENT_NAME!r} was not found")
current_cv_runs = mlflow.search_runs(
    experiment_ids=[cv_experiment.experiment_id],
    filter_string=(
        f"tags.execution_uuid = '{NOTEBOOK_EXECUTION_UUID}' "
        "and tags.phase = 'cv' "
        "and tags.run_type = 'candidate_parent'"
    ),
)
current_fold_runs = mlflow.search_runs(
    experiment_ids=[cv_experiment.experiment_id],
    filter_string=(
        f"tags.execution_uuid = '{NOTEBOOK_EXECUTION_UUID}' "
        "and tags.phase = 'cv' "
        "and tags.run_type = 'fold'"
    ),
)
parent_keys = {
    (
        str(row["tags.subset"]),
        str(row["tags.hidden_layer_sizes"]),
        float(row["params.alpha"]),
    )
    for _, row in current_cv_runs.iterrows()
}
if (
    len(current_cv_runs) != len(expected_candidate_keys)
    or parent_keys != expected_candidate_keys
):
    raise ValueError(
        "Current execution must produce the complete 96-candidate subset/hidden_layer_sizes/alpha product: "
        f"expected {len(expected_candidate_keys)} {sorted(expected_candidate_keys)}, "
        f"got {len(current_cv_runs)} {sorted(parent_keys)}"
    )
expected_fold_count = len(expected_candidate_keys) * N_VALIDATION_FOLDS
if len(current_fold_runs) != expected_fold_count:
    raise ValueError(
        f"Current execution must produce {expected_fold_count} nested fold runs, got {len(current_fold_runs)}"
    )
fold_keys = {
    (
        str(row["tags.subset"]),
        str(row["tags.hidden_layer_sizes"]),
        float(row["params.alpha"]),
        int(row["tags.fold"]),
    )
    for _, row in current_fold_runs.iterrows()
}
expected_fold_keys = {
    (subset_name, hidden_layer_sizes_label, alpha, fold_number)
    for subset_name, hidden_layer_sizes_label, alpha in expected_candidate_keys
    for fold_number in range(1, N_VALIDATION_FOLDS + 1)
}
if fold_keys != expected_fold_keys:
    raise ValueError(
        "Current execution fold runs do not cover every candidate and fold"
    )
cv_results = pd.DataFrame(cv_results_rows)
if len(cv_results) != len(expected_candidate_keys):
    raise ValueError("The in-memory CV result table is incomplete")
if (
    set(
        zip(
            cv_results["subset"],
            cv_results["hidden_layer_sizes"],
            cv_results["alpha"],
        )
    )
    != expected_candidate_keys
):
    raise ValueError(
        "The in-memory CV result table does not match the candidate product"
    )
cv_results = cv_results.sort_values(
    ["subset", "hidden_layer_sizes", "alpha"], kind="stable"
).reset_index(drop=True)
selected_subset, selected_hidden_layer_sizes, selected_alpha = select_candidate(
    cv_results, CV_SELECTION_METRIC
)
selected_hidden_layer_sizes_label = format_hidden_layer_sizes(
    selected_hidden_layer_sizes
)
selected_feature_columns = FEATURE_SUBSETS[selected_subset]
fold_horizon_rows = cv_horizon_rows_by_candidate[
    (selected_subset, selected_hidden_layer_sizes_label, selected_alpha)
]
print(
    f"Selected MLP candidate by CV {CV_SELECTION_METRIC.upper()}: "
    f"{selected_subset!r}, hidden_layer_sizes={selected_hidden_layer_sizes_label}, alpha={selected_alpha:g}"
)
display(
    cv_results[
        [
            "subset",
            "feature_count",
            "hidden_layer_sizes",
            "alpha",
            "mae_mean",
            "mae_std",
            "rmse_mean",
            "rmse_std",
            "me_mean",
            "me_std",
            "r2_mean",
            "r2_std",
        ]
    ]
)

## Retrain the selected subset, hidden_layer_sizes, and alpha

The selected `(feature subset, hidden_layer_sizes, alpha)` triple is retrained once on all eligible, chronologically ordered training rows, using the same `build_mlp_estimator` construction path — and the same fixed `RANDOM_STATE` — as every CV fold. The estimator is fitted on the full eligible training cohort, then the sealed test predictors are scored. The model is held in memory here; it is written to disk together with its manifest only after the sealed-test cell below succeeds, so a crashed run leaves the previous artifacts intact.

In [ ]:
final_model = build_mlp_estimator(
    selected_feature_columns,
    hidden_layer_sizes=selected_hidden_layer_sizes,
    alpha=selected_alpha,
    activation=MLP_ACTIVATION,
    solver=MLP_SOLVER,
    learning_rate_init=MLP_LEARNING_RATE_INIT,
    max_iter=MLP_MAX_ITER,
    batch_size=MLP_BATCH_SIZE,
    early_stopping=MLP_EARLY_STOPPING,
    random_state=RANDOM_STATE,
)
final_model.fit(
    numeric_predictors(train_rows, selected_feature_columns),
    train_rows[TARGET_COLUMNS],
)
test_predictions = validate_predictions(
    final_model.predict(numeric_predictors(test_rows, selected_feature_columns)),
    expected_rows=len(test_rows),
    target_columns=TARGET_COLUMNS,
    artifact_name="test",
)
print(
    f"Fitted the selected MLP candidate on {len(train_rows):,} eligible training "
    f"rows and scored {len(test_rows):,} sealed-test rows."
)

## Evaluate on the test cohort

A single scoring pass over the sealed test cohort reports aggregate MAE/RMSE, the same metrics for each lead in the direct 24-hour forecast, and a short preview for comparison with actual targets. Plot and MLflow labels identify the selected subset, hidden_layer_sizes, and alpha. There is no second pass and no refitting.

The model and its `schema_version: "1.0"` manifest are written at the end of this cell. The manifest is the durable record of this execution — the selected subset, hidden_layer_sizes, and alpha, the full 96-candidate CV table, the aggregate sealed-test metrics, and the per-horizon sealed-test metrics — and it is what the evaluation section below reads. MLflow still receives the same params, metrics, and figures; it remains the run log and UI, not the record the notebook reads back.

In [ ]:
aggregate_metrics, per_horizon_metrics = metric_tables(
    test_rows[TARGET_COLUMNS],
    test_predictions,
    target_columns=TARGET_COLUMNS,
    station_id=station_id,
)
if not np.isfinite(aggregate_metrics[["mae", "rmse", "me", "r2"]].to_numpy()).all():
    raise ValueError("MLP reported non-finite aggregate metrics")
if not np.isfinite(per_horizon_metrics[["mae", "rmse", "me", "r2"]].to_numpy()).all():
    raise ValueError("MLP reported non-finite horizon metrics")
with mlflow.start_run(
    run_name=f"mlp_test_{selected_subset}_{selected_hidden_layer_sizes_label}_alpha_{selected_alpha:g}",
    nested=False,
    tags={
        "phase": "test",
        "run_type": "sealed_test",
        "subset": selected_subset,
        "execution_uuid": NOTEBOOK_EXECUTION_UUID,
    },
):
    mlflow.log_params(
        {
            "phase": "test",
            "run_type": "sealed_test",
            "subset": selected_subset,
            "feature_count": len(selected_feature_columns),
            "feature_columns": json.dumps(selected_feature_columns),
            **INPUT_PARQUET_SHA256_PARAMS,
            "alpha": selected_alpha,
            "hidden_layer_sizes": selected_hidden_layer_sizes_label,
            "selection_metric": CV_SELECTION_METRIC,
            "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
            "cv_selected_metric": float(
                cv_results.loc[
                    cv_results["subset"].eq(selected_subset)
                    & cv_results["hidden_layer_sizes"].eq(
                        selected_hidden_layer_sizes_label
                    )
                    & cv_results["alpha"].eq(selected_alpha),
                    f"{CV_SELECTION_METRIC}_mean",
                ].iloc[0]
            ),
            "scored_issue_times": len(test_rows),
        }
    )
    mlflow.log_metrics(
        {
            "test_mae": float(aggregate_metrics.iloc[0]["mae"]),
            "test_rmse": float(aggregate_metrics.iloc[0]["rmse"]),
            "test_me": float(aggregate_metrics.iloc[0]["me"]),
            "test_r2": float(aggregate_metrics.iloc[0]["r2"]),
            **{
                f"test_mae_horizon_{row.horizon_hours:02d}": float(row.mae)
                for row in per_horizon_metrics.itertuples()
            },
            **{
                f"test_me_horizon_{row.horizon_hours:02d}": float(row.me)
                for row in per_horizon_metrics.itertuples()
            },
            **{
                f"test_r2_horizon_{row.horizon_hours:02d}": float(row.r2)
                for row in per_horizon_metrics.itertuples()
            },
            **{
                f"test_rmse_horizon_{row.horizon_hours:02d}": float(row.rmse)
                for row in per_horizon_metrics.itertuples()
            },
        }
    )
    cv_horizon_metrics = pd.concat(fold_horizon_rows, ignore_index=True)
    cv_rmse_mae_boxplots_fig = cv_error_boxplots_figure(
        cv_horizon_metrics,
        TARGET_COLUMNS,
        title=f"MLP CV errors — {selected_subset}, hidden_layer_sizes={selected_hidden_layer_sizes_label}, alpha={selected_alpha:g}",
    )
    mlflow.log_figure(cv_rmse_mae_boxplots_fig, "cv_rmse_mae_boxplots.png")
    plt.show()
    plt.close(cv_rmse_mae_boxplots_fig)
    test_error_boxplots_fig = test_error_boxplots_figure(
        test_rows,
        test_predictions,
        per_horizon_metrics,
        TARGET_COLUMNS,
        title=f"MLP final-test errors — {selected_subset}, hidden_layer_sizes={selected_hidden_layer_sizes_label}, alpha={selected_alpha:g}",
    )
    mlflow.log_figure(test_error_boxplots_fig, "test_error_boxplots.png")
    plt.show()
    plt.close(test_error_boxplots_fig)
    test_predicted_vs_actual_fig = predicted_vs_actual_figure(
        test_rows[TARGET_COLUMNS],
        test_predictions,
        TARGET_COLUMNS,
        title=f"MLP predicted vs actual — {selected_subset}, hidden_layer_sizes={selected_hidden_layer_sizes_label}, alpha={selected_alpha:g}",
    )
    mlflow.log_figure(test_predicted_vs_actual_fig, "test_predicted_vs_actual.png")
    plt.show()
    plt.close(test_predicted_vs_actual_fig)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
dump(final_model, MODEL_PATH)
save_mlp_manifest(
    MODEL_METADATA_PATH,
    model_path=MODEL_PATH,
    execution_uuid=NOTEBOOK_EXECUTION_UUID,
    contract=contract,
    feature_subsets=FEATURE_SUBSETS,
    selected_subset=selected_subset,
    selected_hidden_layer_sizes=selected_hidden_layer_sizes,
    selected_alpha=selected_alpha,
    selection_metric=CV_SELECTION_METRIC,
    cv_results=cv_results,
    sealed_test_metrics={
        "test_mae": float(aggregate_metrics.iloc[0]["mae"]),
        "test_rmse": float(aggregate_metrics.iloc[0]["rmse"]),
        "test_me": float(aggregate_metrics.iloc[0]["me"]),
        "test_r2": float(aggregate_metrics.iloc[0]["r2"]),
    },
    per_horizon_metrics=per_horizon_metrics,
    cohort={
        "contract": "full_feature_columns",
        "rule": "target_valid and complete full predictor and target contract",
        "train_raw_rows": dataset.raw_row_counts["train"],
        "train_eligible_rows": len(train_rows),
        "test_raw_rows": dataset.raw_row_counts["test"],
        "test_eligible_rows": len(test_rows),
        "same_folds_for_all_candidates": True,
    },
    training={
        "source_artifact": str(train_path),
        "raw_rows": dataset.raw_row_counts["train"],
        "eligible_rows": len(train_rows),
        "eligibility": "target_valid and complete full predictor and target contract",
        "timestamp_start": train_rows["timestamp"].iloc[0].isoformat(),
        "timestamp_end": train_rows["timestamp"].iloc[-1].isoformat(),
    },
)
print(f"Saved MLP model to {MODEL_PATH}")
print(f"Saved MLP model manifest to {MODEL_METADATA_PATH}")
print(
    f"MLP test results for {station_id} "
    f"(selected subset={selected_subset!r}, hidden_layer_sizes={selected_hidden_layer_sizes_label}, alpha={selected_alpha:g})"
)
display(aggregate_metrics)
display(per_horizon_metrics)
display(
    prediction_preview(
        test_rows,
        test_predictions,
        target_columns=TARGET_COLUMNS,
    ).head(PREDICTION_PREVIEW_ROWS)
)

# MLP saved-model evaluation

This read-only section reads the saved MLP manifest, `models/mlp_{TARGET_STATION_ID}.json`, which the sealed-test cell above writes once per successful execution. It compares that execution's feature-subset/hidden_layer_sizes/alpha candidates using their cross-validation metrics and reports sealed-test metrics only for the selected candidate. It does not query MLflow: the manifest is the record, MLflow is the run log.

## Load the saved MLP execution record

The joined dataset and the manifest are loaded here, so this section does not depend on any variable created by the training cells and can be re-run on its own in a fresh kernel. `load_mlp_manifest()` checks the manifest against the *current* feature contract — station, horizon, full predictor columns, target columns, and the selected subset's columns — so a Stage-3 re-run that changes the contract fails here instead of silently scoring a stale model.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

from src.config import (
    CV_SELECTION_METRIC,
    EMBARGO_HOURS,
    FORECAST_HORIZON_HOURS,
    INITIAL_TRAIN_FRACTION,
    N_VALIDATION_FOLDS,
    TARGET_STATION_ID,
    WEATHER_VARIABLES,
)
from src.dataset import load_joined_dataset
from src.mlp import format_hidden_layer_sizes, load_mlp_manifest, score_saved_model
from src.plots import forecast_window_figures

if CV_SELECTION_METRIC not in {"mae", "rmse"}:
    raise ValueError("CV_SELECTION_METRIC must be either 'mae' or 'rmse'")

COMPARISON_PROCESSED_DIR = Path("data/processed/joined")
COMPARISON_METADATA_PATH = (
    COMPARISON_PROCESSED_DIR / "all_stations_feature_metadata.json"
)
COMPARISON_TRAIN_PATH = COMPARISON_PROCESSED_DIR / "all_stations_train_features.parquet"
COMPARISON_TEST_PATH = COMPARISON_PROCESSED_DIR / "all_stations_test_features.parquet"
COMPARISON_MODEL_PATH = Path("models") / f"mlp_{TARGET_STATION_ID}.joblib"
COMPARISON_MODEL_METADATA_PATH = Path("models") / f"mlp_{TARGET_STATION_ID}.json"

comparison_dataset = load_joined_dataset(
    COMPARISON_METADATA_PATH,
    COMPARISON_TRAIN_PATH,
    COMPARISON_TEST_PATH,
    station_id=TARGET_STATION_ID,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
    weather_variables=WEATHER_VARIABLES,
    initial_train_fraction=INITIAL_TRAIN_FRACTION,
    n_validation_folds=N_VALIDATION_FOLDS,
    embargo_rows=EMBARGO_HOURS,
)
comparison_contract = comparison_dataset.contract
COMPARISON_TARGET_COLUMNS = list(comparison_contract.target_columns)
comparison_test_rows = comparison_dataset.test_rows

mlp_manifest = load_mlp_manifest(
    COMPARISON_MODEL_METADATA_PATH,
    contract=comparison_contract,
    feature_subsets=comparison_dataset.feature_subsets,
)

## Inspect the recorded execution

The manifest is written once, after the sealed test has been scored, so an execution that crashed part-way leaves no record and the previous manifest survives untouched. The `execution_uuid` below is the same tag the MLflow runs of that execution carry, which is how the two views are tied together.

In [ ]:
selected_candidate_table = mlp_manifest.cv_results
selected_horizon_metrics = mlp_manifest.horizon_metrics
selected_hidden_layer_sizes_label = format_hidden_layer_sizes(
    mlp_manifest.selected_hidden_layer_sizes
)
selected_candidate = selected_candidate_table.loc[
    selected_candidate_table["subset"].eq(mlp_manifest.selected_subset)
    & selected_candidate_table["hidden_layer_sizes"].eq(
        selected_hidden_layer_sizes_label
    )
    & selected_candidate_table["alpha"].eq(mlp_manifest.selected_alpha)
].iloc[0]
selected_execution_summary = pd.DataFrame(
    [
        {
            "execution_uuid": mlp_manifest.execution_uuid,
            "manifest": str(COMPARISON_MODEL_METADATA_PATH),
            "candidate_count": len(selected_candidate_table),
            "selection_metric": mlp_manifest.selection_metric,
            "selected_subset": mlp_manifest.selected_subset,
            "selected_hidden_layer_sizes": selected_hidden_layer_sizes_label,
            "selected_alpha": mlp_manifest.selected_alpha,
        }
    ]
)
display(selected_execution_summary)

## Compare cross-validation candidates

Candidate ranking uses only the recorded CV metrics and the configured selection metric; the sealed-test metrics are not used to rank candidates.

In [ ]:
candidate_columns = [
    "subset",
    "feature_count",
    "hidden_layer_sizes",
    "alpha",
    "mae_mean",
    "mae_std",
    "rmse_mean",
    "rmse_std",
    "me_mean",
    "me_std",
    "r2_mean",
    "r2_std",
]
candidate_comparison_table = selected_candidate_table[candidate_columns].copy()
display(candidate_comparison_table)

## Visualize cross-validation error

The heatmap shows CV RMSE across feature subsets and alpha values, faceted by hidden_layer_sizes. The line chart adds fold-to-fold RMSE variation as error bars, with alpha on a log scale.

In [ ]:
HIDDEN_LAYER_SIZE_FACETS = ["32", "64", "128", "64x32"]
cv_rmse_heatmap_zmin = candidate_comparison_table["rmse_mean"].min()
cv_rmse_heatmap_zmax = candidate_comparison_table["rmse_mean"].max()
cv_rmse_heatmap_figure = make_subplots(
    rows=1,
    cols=len(HIDDEN_LAYER_SIZE_FACETS),
    subplot_titles=[
        f"hidden_layer_sizes={label}" for label in HIDDEN_LAYER_SIZE_FACETS
    ],
    shared_yaxes=True,
)
for column_index, hidden_layer_sizes_label in enumerate(
    HIDDEN_LAYER_SIZE_FACETS, start=1
):
    facet_table = candidate_comparison_table[
        candidate_comparison_table["hidden_layer_sizes"].eq(hidden_layer_sizes_label)
    ]
    facet_heatmap_values = (
        facet_table.pivot(index="subset", columns="alpha", values="rmse_mean")
        .sort_index(axis=0)
        .sort_index(axis=1)
    )
    cv_rmse_heatmap_figure.add_trace(
        go.Heatmap(
            z=facet_heatmap_values.to_numpy(),
            x=list(map(str, facet_heatmap_values.columns.tolist())),
            y=facet_heatmap_values.index.tolist(),
            zmin=cv_rmse_heatmap_zmin,
            zmax=cv_rmse_heatmap_zmax,
            coloraxis="coloraxis",
            hovertemplate="Subset=%{y}<br>Alpha=%{x}<br>CV RMSE=%{z:.4f}<extra></extra>",
        ),
        row=1,
        col=column_index,
    )
cv_rmse_heatmap_figure.update_layout(
    title="MLP candidate CV RMSE by feature subset, alpha, and hidden_layer_sizes",
    coloraxis={"colorscale": "Viridis", "colorbar": {"title": "CV RMSE"}},
)
cv_rmse_heatmap_figure.update_xaxes(title_text="Alpha")
cv_rmse_heatmap_figure.update_yaxes(title_text="Feature subset", col=1)
display(cv_rmse_heatmap_figure)

In [ ]:
cv_rmse_by_alpha_figure = go.Figure()
for (
    subset_name,
    hidden_layer_sizes_label,
), subset_candidates in candidate_comparison_table.groupby(
    ["subset", "hidden_layer_sizes"], sort=True
):
    subset_candidates = subset_candidates.sort_values("alpha")
    cv_rmse_by_alpha_figure.add_trace(
        go.Scatter(
            x=subset_candidates["alpha"],
            y=subset_candidates["rmse_mean"],
            mode="lines+markers",
            name=f"{subset_name} · {hidden_layer_sizes_label}",
            error_y={
                "type": "data",
                "array": subset_candidates["rmse_std"],
                "visible": True,
            },
        )
    )
cv_rmse_by_alpha_figure.update_layout(
    title="MLP CV RMSE versus alpha, by feature subset and hidden_layer_sizes",
    xaxis_title="Alpha",
    yaxis_title="CV RMSE",
    xaxis_type="log",
)
display(cv_rmse_by_alpha_figure)

## Inspect selected-candidate sealed-test performance

These values belong only to the candidate recorded by the selected sealed-test run. The final chart shows its MAE and RMSE at each available forecast horizon.

In [ ]:
selected_candidate_sealed_test_summary = pd.DataFrame(
    [
        {
            "execution_uuid": mlp_manifest.execution_uuid,
            "subset": selected_candidate["subset"],
            "feature_count": selected_candidate["feature_count"],
            "hidden_layer_sizes": selected_candidate["hidden_layer_sizes"],
            "alpha": selected_candidate["alpha"],
            "cv_mae_mean": selected_candidate["mae_mean"],
            "cv_rmse_mean": selected_candidate["rmse_mean"],
            "cv_me_mean": selected_candidate["me_mean"],
            "cv_r2_mean": selected_candidate["r2_mean"],
            **mlp_manifest.sealed_test_metrics,
        }
    ]
)
display(selected_candidate_sealed_test_summary)

sealed_test_horizon_figure = go.Figure(
    [
        go.Scatter(
            x=selected_horizon_metrics["horizon_hours"],
            y=selected_horizon_metrics[metric_name],
            mode="lines+markers",
            name=metric_name.upper(),
        )
        for metric_name in ("test_mae", "test_rmse", "test_me", "test_r2")
    ]
)
sealed_test_horizon_figure.update_layout(
    title="Selected MLP candidate sealed-test error by horizon",
    xaxis_title="Forecast horizon (hours)",
    yaxis_title="Error",
)
display(sealed_test_horizon_figure)

## Reload and score the saved MLP model

Reloads the saved MLP model and scores the eligible sealed-test cohort on the manifest's selected feature columns, without retraining or changing the stored prediction semantics.

In [ ]:
comparison_prediction_values = score_saved_model(
    mlp_manifest,
    COMPARISON_MODEL_PATH,
    comparison_test_rows,
)
print(
    f"Scored {len(comparison_test_rows):,} eligible sealed-test rows with "
    f"{len(COMPARISON_TARGET_COLUMNS)} horizons using the saved MLP model."
)

In [ ]:
comparison_prediction_columns = [
    f"prediction_{target_column}" for target_column in COMPARISON_TARGET_COLUMNS
]
comparison_prediction_table = (
    comparison_test_rows[["timestamp", *COMPARISON_TARGET_COLUMNS]]
    .reset_index(drop=True)
    .rename(columns={"timestamp": "issue_time"})
)
comparison_prediction_table = pd.concat(
    [
        comparison_prediction_table,
        pd.DataFrame(
            comparison_prediction_values,
            columns=comparison_prediction_columns,
        ),
    ],
    axis=1,
)
comparison_prediction_table["issue_time"] = pd.to_datetime(
    comparison_prediction_table["issue_time"], utc=True
)

comparison_issue_times = comparison_prediction_table["issue_time"]
comparison_horizons = list(range(1, FORECAST_HORIZON_HOURS + 1))
comparison_horizon_labels = [f"H+{horizon:02d}" for horizon in comparison_horizons]
comparison_time_series_frames = []
for horizon in comparison_horizons:
    target_column = COMPARISON_TARGET_COLUMNS[horizon - 1]
    prediction_column = comparison_prediction_columns[horizon - 1]
    valid_times = comparison_issue_times + pd.to_timedelta(horizon, unit="h")
    comparison_time_series_frames.append(
        go.Frame(
            name=comparison_horizon_labels[horizon - 1],
            data=[
                go.Scattergl(
                    x=valid_times,
                    y=comparison_prediction_table[target_column],
                    customdata=comparison_issue_times,
                    mode="lines+markers",
                    name="Actual",
                    hovertemplate="Valid time=%{x}<br>Issue time=%{customdata}<br>Actual=%{y:.3f}<extra></extra>",
                ),
                go.Scattergl(
                    x=valid_times,
                    y=comparison_prediction_table[prediction_column],
                    customdata=comparison_issue_times,
                    mode="lines+markers",
                    name="Prediction",
                    hovertemplate="Valid time=%{x}<br>Issue time=%{customdata}<br>Prediction=%{y:.3f}<extra></extra>",
                ),
            ],
        )
    )
comparison_time_series_steps = [
    {
        "label": comparison_horizon_labels[horizon - 1],
        "method": "animate",
        "args": [[comparison_horizon_labels[horizon - 1]], {"mode": "immediate"}],
    }
    for horizon in comparison_horizons
]
comparison_time_series_figure = go.Figure(
    data=comparison_time_series_frames[0].data,
    frames=comparison_time_series_frames,
    layout={
        "title": "Saved MLP predictions across forecast horizons",
        "xaxis_title": "Valid time",
        "yaxis_title": "Water level",
        "hovermode": "x unified",
        "sliders": [
            {
                "active": 0,
                "currentvalue": {"prefix": "Forecast horizon: "},
                "steps": comparison_time_series_steps,
            }
        ],
    },
)
mlp_prediction_time_series_figure = comparison_time_series_figure
display(comparison_time_series_figure)

## Inspect best and worst MLP forecast windows

The following plots use the saved-model predictions and select sealed-test issue times by the RMSE calculated across all 24 forecast horizons. A context window is eligible only when the target-station water-level series contains every hourly observation from 48 hours before through 48 hours after the issue time, with no imputed observations.

In [ ]:
mlp_forecast_window_figures = forecast_window_figures(
    comparison_prediction_table,
    comparison_dataset.target_context_series,
    water_level_column=f"{TARGET_STATION_ID}__water_level",
    imputed_column=f"{TARGET_STATION_ID}__imputed",
    prediction_columns=comparison_prediction_columns,
    target_columns=COMPARISON_TARGET_COLUMNS,
    horizons=comparison_horizons,
    label_prefix="MLP",
)

### Best

In [ ]:
best_mlp_forecast_window_figure = mlp_forecast_window_figures["best"]
display(best_mlp_forecast_window_figure)

### Worst

In [ ]:
worst_mlp_forecast_window_figure = mlp_forecast_window_figures["worst"]
display(worst_mlp_forecast_window_figure)

## Compare absolute and signed errors

Each box contains all eligible sealed-test errors for one horizon. Absolute-error markers reuse the manifest's per-horizon MAE/RMSE values; signed errors follow the convention `prediction - actual`.

In [ ]:
comparison_actual_values = comparison_test_rows[COMPARISON_TARGET_COLUMNS].to_numpy(
    dtype=float
)
signed_errors = comparison_prediction_values - comparison_actual_values
absolute_errors = np.abs(signed_errors)

absolute_error_boxplot_figure = go.Figure(
    data=[
        go.Box(
            x=comparison_horizon_labels * len(absolute_errors),
            y=absolute_errors.reshape(-1),
            name="Boxplots",
            boxpoints=False,
        ),
        go.Scatter(
            x=comparison_horizon_labels,
            y=selected_horizon_metrics["test_mae"],
            mode="markers",
            name="MAE",
        ),
        go.Scatter(
            x=comparison_horizon_labels,
            y=selected_horizon_metrics["test_rmse"],
            mode="markers",
            name="RMSE",
        ),
    ]
)
absolute_error_boxplot_figure.update_layout(
    title="Saved MLP absolute errors by forecast horizon",
    xaxis={
        "title": "Forecast horizon",
        "type": "category",
        "categoryorder": "array",
        "categoryarray": comparison_horizon_labels,
    },
    yaxis_title="Absolute error",
)

display(absolute_error_boxplot_figure)

In [ ]:
signed_error_boxplot_figure = go.Figure(
    data=[
        go.Box(
            x=comparison_horizon_labels * len(signed_errors),
            y=signed_errors.reshape(-1),
            name="Boxplots",
            boxpoints=False,
        ),
        go.Scatter(
            x=comparison_horizon_labels,
            y=signed_errors.mean(axis=0),
            mode="markers",
            name="Mean error",
        ),
    ]
)
signed_error_boxplot_figure.update_layout(
    title="Saved MLP signed errors by forecast horizon",
    xaxis={
        "title": "Forecast horizon",
        "type": "category",
        "categoryorder": "array",
        "categoryarray": comparison_horizon_labels,
    },
    yaxis_title="Signed error (prediction - actual)",
)
signed_error_boxplot_figure.add_hline(
    y=0,
    line_dash="dash",
    line_color="black",
)
display(signed_error_boxplot_figure)

## Selected MLP architecture

Ridge's final section extracts a closed-form linear coefficient formula, which has no counterpart for a nonlinear MLP. Instead, this section reloads the saved model and reports its fitted architecture, parameter count, and whether training converged before the fixed `max_iter` cap.

In [ ]:
from IPython.display import Markdown
from joblib import load as load_joblib

architecture_model = load_joblib(COMPARISON_MODEL_PATH)
architecture_regressor = getattr(architecture_model, "regressor_", architecture_model)
architecture_mlp = architecture_regressor.named_steps["mlp"]
architecture_parameter_count = sum(
    weights.size for weights in architecture_mlp.coefs_
) + sum(biases.size for biases in architecture_mlp.intercepts_)
architecture_hit_iteration_cap = architecture_mlp.n_iter_ >= architecture_mlp.max_iter

architecture_summary_table = pd.DataFrame(
    [
        {
            "hidden_layer_sizes": selected_hidden_layer_sizes_label,
            "alpha": mlp_manifest.selected_alpha,
            "activation": architecture_mlp.activation,
            "solver": architecture_mlp.solver,
            "parameter_count": architecture_parameter_count,
            "n_iter_": architecture_mlp.n_iter_,
            "max_iter": architecture_mlp.max_iter,
            "hit_iteration_cap": architecture_hit_iteration_cap,
        }
    ]
)
display(
    Markdown(
        f"""The saved model uses subset **{mlp_manifest.selected_subset}**,
hidden_layer_sizes **{selected_hidden_layer_sizes_label}**, and
alpha **{mlp_manifest.selected_alpha:g}**."""
    )
)
display(architecture_summary_table)
if architecture_hit_iteration_cap:
    print(
        "Training hit the fixed max_iter cap rather than converging "
        "(early_stopping=False); note this in the thesis text."
    )